In [10]:
# ============================================
# CELL 1: Imports & Configuration
# ============================================

import os
import json
import yaml
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import models, transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Image Processing
import cv2
from PIL import Image

# ML Utilities
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)

# Experiment Tracking
import mlflow
import mlflow.pytorch

# Settings
plt.style.use('seaborn-v0_8')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ Libraries imported!")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

# ============================================
# TRAINING CONFIGURATION
# ============================================

CONFIG = {
    # Paths
    'train_csv':        '../data/processed/train_clean.csv',
    'test_csv':         '../data/processed/test_clean.csv',
    'val_csv':          '../data/processed/val_clean.csv',
    'weights_path':     '../configs/class_weights.json',
    'model_save_path':  '../experiments/best_model.pth',
    'last_model_path':  '../experiments/last_model.pth',

    # Image settings
    'image_size':       224,
    'channels':         3,

    # Training settings
    'batch_size':       8,
    'num_workers':      0,
    'pin_memory':       True,
    'epochs':           25,
    'learning_rate':    0.001,
    'weight_decay':     1e-4,
    'patience':         5,       # Early stopping patience

    # Model settings
    'model_name':       'efficientnet_b0',
    'num_classes':      2,
    'pretrained':       True,
    'dropout_rate':     0.3,

    # Normalization
    'mean': [0.485, 0.456, 0.406],
    'std':  [0.229, 0.224, 0.225],

    # Classes
    'classes':          ['NORMAL', 'PNEUMONIA'],
}

print(f"\n📋 Training Configuration:")
print(f"  Model:          {CONFIG['model_name']}")
print(f"  Epochs:         {CONFIG['epochs']}")
print(f"  Batch Size:     {CONFIG['batch_size']}")
print(f"  Learning Rate:  {CONFIG['learning_rate']}")
print(f"  Early Stopping: {CONFIG['patience']} epochs patience")
print(f"  Device:         {device}")

✅ Libraries imported!
Device: cuda
PyTorch: 2.7.1+cu118

📋 Training Configuration:
  Model:          efficientnet_b0
  Epochs:         25
  Batch Size:     8
  Learning Rate:  0.001
  Early Stopping: 5 epochs patience
  Device:         cuda


In [11]:
# ============================================
# CELL 2: Dataset & DataLoaders
# ============================================

# Load data
train_df = pd.read_csv(CONFIG['train_csv'])
test_df  = pd.read_csv(CONFIG['test_csv'])
val_df   = pd.read_csv(CONFIG['val_csv'])

with open(CONFIG['weights_path'], 'r') as f:
    class_weights = json.load(f)

print("✅ Data loaded!")

# CLAHE function
def apply_clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l)
    lab_enhanced = cv2.merge([l_enhanced, a, b])
    return cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2RGB)

# Transforms
train_transforms = A.Compose([
    A.Resize(CONFIG['image_size'], CONFIG['image_size']),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
    A.ElasticTransform(alpha=1, sigma=50, p=0.2),
    A.CoarseDropout(num_holes_range=(1,8),
                    hole_height_range=(8,16),
                    hole_width_range=(8,16), p=0.3),
    A.Normalize(mean=CONFIG['mean'], std=CONFIG['std']),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(CONFIG['image_size'], CONFIG['image_size']),
    A.Normalize(mean=CONFIG['mean'], std=CONFIG['std']),
    ToTensorV2()
])

# Dataset class
class ChestXRayDataset(Dataset):
    def __init__(self, dataframe, transforms=None, apply_clahe_flag=True):
        self.df = dataframe.reset_index(drop=True)
        self.transforms = transforms
        self.apply_clahe_flag = apply_clahe_flag
        self.class_to_idx = {cls: idx for idx, cls 
                             in enumerate(CONFIG['classes'])}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['filepath'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.apply_clahe_flag:
            image = apply_clahe(image)
        if self.transforms:
            image = self.transforms(image=image)['image']
        label = self.class_to_idx[row['label']]
        return image, torch.tensor(label, dtype=torch.long)

    def get_labels(self):
        return [self.class_to_idx[l] for l in self.df['label']]

# Create datasets
train_dataset = ChestXRayDataset(train_df, train_transforms)
val_dataset   = ChestXRayDataset(val_df,   val_transforms)
test_dataset  = ChestXRayDataset(test_df,  val_transforms)

# WeightedRandomSampler
train_labels    = train_dataset.get_labels()
sample_weights  = torch.tensor([
    class_weights['NORMAL'] if l == 0 else class_weights['PNEUMONIA']
    for l in train_labels], dtype=torch.float)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          sampler=sampler, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'],
                          shuffle=False,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG['batch_size'],
                          shuffle=False,  num_workers=0, pin_memory=True)

print(f"✅ DataLoaders ready!")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")

✅ Data loaded!
✅ DataLoaders ready!
  Train batches: 649
  Val batches:   2
  Test batches:  78


In [12]:
# ============================================
# CELL 3: Build EfficientNet Model
# ============================================

class ChestXRayModel(nn.Module):
    """
    EfficientNetB0 based model for Chest X-Ray classification
    Uses transfer learning from ImageNet pretrained weights
    """
    
    def __init__(self, num_classes=2, dropout_rate=0.3, pretrained=True):
        super(ChestXRayModel, self).__init__()
        
        # Load pretrained EfficientNetB0
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.efficientnet_b0(weights=weights)
        
        # Get number of features from last layer
        in_features = self.backbone.classifier[1].in_features
        
        # Replace classifier with custom head
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.backbone(x)


# Create model
model = ChestXRayModel(
    num_classes=CONFIG['num_classes'],
    dropout_rate=CONFIG['dropout_rate'],
    pretrained=CONFIG['pretrained']
)

# Move model to GPU
model = model.to(device)

# Print model summary
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✅ Model created!")
print(f"\n📊 Model Summary:")
print(f"  Architecture:      EfficientNetB0")
print(f"  Total params:      {total_params:,}")
print(f"  Trainable params:  {trainable_params:,}")
print(f"  Pretrained:        {CONFIG['pretrained']}")
print(f"  Dropout rate:      {CONFIG['dropout_rate']}")
print(f"  Output classes:    {CONFIG['num_classes']}")

# Test forward pass
print(f"\n🔍 Testing forward pass...")
dummy_input = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    output = model(dummy_input)
print(f"  Input shape:  {dummy_input.shape}")
print(f"  Output shape: {output.shape}")
print(f"  Output:       {output}")
print(f"\n✅ Forward pass successful!")

✅ Model created!

📊 Model Summary:
  Architecture:      EfficientNetB0
  Total params:      4,336,510
  Trainable params:  4,336,510
  Pretrained:        True
  Dropout rate:      0.3
  Output classes:    2

🔍 Testing forward pass...
  Input shape:  torch.Size([2, 3, 224, 224])
  Output shape: torch.Size([2, 2])
  Output:       tensor([[ 0.2897,  0.2099],
        [-0.8354, -0.7942]], device='cuda:0')

✅ Forward pass successful!


In [13]:
# ============================================
# CELL 4: Loss Function, Optimizer & Scheduler
# ============================================

# ---- Class weights tensor for loss function ----
weights_tensor = torch.tensor([
    class_weights['NORMAL'],
    class_weights['PNEUMONIA']
], dtype=torch.float).to(device)

# ---- Loss Function ----
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

# ---- Optimizer ----
optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

# ---- Learning Rate Scheduler ----
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

# ---- Early Stopping Class ----
class EarlyStopping:
    """
    Stops training when validation loss stops improving
    Saves best model automatically
    """
    def __init__(self, patience=5, min_delta=0.001):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = float('inf')
        self.early_stop = False
        self.best_model = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            # Improvement found
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_model = copy.deepcopy(model.state_dict())
            print(f"    ✅ Val loss improved to {val_loss:.4f} — saving model")
        else:
            # No improvement
            self.counter += 1
            print(f"    ⚠️ No improvement. Counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"    🛑 Early stopping triggered!")

early_stopping = EarlyStopping(
    patience=CONFIG['patience'],
    min_delta=0.001
)

print("✅ Training components ready!")
print(f"\n📋 Setup Summary:")
print(f"  Loss:      CrossEntropyLoss (weighted)")
print(f"  Optimizer: Adam (lr={CONFIG['learning_rate']}, wd={CONFIG['weight_decay']})")
print(f"  Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)")
print(f"  Early Stop: patience={CONFIG['patience']}")
print(f"\n⚖️ Loss weights:")
print(f"  NORMAL:    {class_weights['NORMAL']:.4f}")
print(f"  PNEUMONIA: {class_weights['PNEUMONIA']:.4f}")

✅ Training components ready!

📋 Setup Summary:
  Loss:      CrossEntropyLoss (weighted)
  Optimizer: Adam (lr=0.001, wd=0.0001)
  Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)
  Early Stop: patience=5

⚖️ Loss weights:
  NORMAL:    1.9366
  PNEUMONIA: 0.6740


In [14]:
# ============================================
# CELL 5: Training Loop (Memory Optimized)
# ============================================
from torch.cuda.amp import autocast, GradScaler

# Mixed precision scaler
scaler = GradScaler()

def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train model for one epoch with mixed precision"""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass
        with autocast():
            outputs = model(images)
            loss    = criterion(outputs, labels)

        # Scaled backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted  = outputs.max(1)
        total        += labels.size(0)
        correct      += predicted.eq(labels).sum().item()

        if (batch_idx + 1) % 50 == 0:
            print(f"    Batch {batch_idx+1}/{len(loader)} "
                  f"Loss: {running_loss/(batch_idx+1):.4f} "
                  f"Acc: {100.*correct/total:.2f}%")

    return running_loss/len(loader), 100.*correct/total


def validate(model, loader, criterion, device):
    """Validate model"""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            with autocast():
                outputs = model(images)
                loss    = criterion(outputs, labels)

            probs        = torch.softmax(outputs.float(), dim=1)
            running_loss += loss.item()
            _, predicted  = outputs.max(1)
            total        += labels.size(0)
            correct      += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_acc  = 100. * correct / total
    auc        = roc_auc_score(all_labels, all_probs)
    return epoch_loss, epoch_acc, auc


# ============================================
# MAIN TRAINING LOOP WITH MLFLOW TRACKING
# ============================================

os.makedirs('../experiments', exist_ok=True)
mlflow.set_tracking_uri('../experiments/mlruns')
mlflow.set_experiment('chest-xray-pneumonia')

# Clear GPU cache before training
torch.cuda.empty_cache()

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   [], 'val_auc': []
}

print("🚀 Starting Training (Mixed Precision)...")
print("=" * 60)

with mlflow.start_run(run_name='efficientnet_b0_v1'):

    mlflow.log_params({
        'model':         CONFIG['model_name'],
        'epochs':        CONFIG['epochs'],
        'batch_size':    CONFIG['batch_size'],
        'learning_rate': CONFIG['learning_rate'],
        'weight_decay':  CONFIG['weight_decay'],
        'image_size':    CONFIG['image_size'],
        'dropout_rate':  CONFIG['dropout_rate'],
        'pretrained':    CONFIG['pretrained'],
    })

    best_val_acc = 0.0
    start_time   = time.time()

    for epoch in range(CONFIG['epochs']):
        epoch_start = time.time()
        print(f"\n📌 Epoch {epoch+1}/{CONFIG['epochs']}")
        print("-" * 40)

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device)

        val_loss, val_acc, val_auc = validate(
            model, val_loader, criterion, device)

        scheduler.step(val_loss)
        early_stopping(val_loss, model)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)

        mlflow.log_metrics({
            'train_loss': train_loss,
            'train_acc':  train_acc,
            'val_loss':   val_loss,
            'val_acc':    val_acc,
            'val_auc':    val_auc,
        }, step=epoch)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch':       epoch,
                'model_state': model.state_dict(),
                'optimizer':   optimizer.state_dict(),
                'val_acc':     val_acc,
                'val_auc':     val_auc,
                'config':      CONFIG,
            }, CONFIG['model_save_path'])

        epoch_time = time.time() - epoch_start
        print(f"\n  📊 Epoch {epoch+1} Results:")
        print(f"     Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"     Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}% | AUC: {val_auc:.4f}")
        print(f"     Best Val Acc: {best_val_acc:.2f}%")
        print(f"     Time: {epoch_time:.1f}s")

        # Clear cache every epoch
        torch.cuda.empty_cache()

        if early_stopping.early_stop:
            print(f"\n🛑 Early stopping at epoch {epoch+1}")
            break

    checkpoint = torch.load(CONFIG['model_save_path'])
    model.load_state_dict(checkpoint['model_state'])

    total_time = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"🎉 Training Complete!")
    print(f"   Best Val Acc: {best_val_acc:.2f}%")
    print(f"   Total Time:   {total_time/60:.1f} minutes")

    mlflow.log_metrics({
        'best_val_acc':   best_val_acc,
        'total_time_min': total_time/60
    })
    mlflow.pytorch.log_model(model, 'model')

🚀 Starting Training (Mixed Precision)...

📌 Epoch 1/25
----------------------------------------


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 3.00 GiB of which 917.85 MiB is free. Of the allocated memory 1.45 GiB is allocated by PyTorch, and 62.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)